# Lesson 4.5 — Event-Driven Automation: Lambda, S3 Events & EventBridge

**Level: Intermediate** | **Exam:** SOA-C03 Domain 3, Skill 3.2.2

## Learning Objectives
- Trigger Lambda functions from S3 events (object creation, deletion)
- Build EventBridge rules that invoke Lambda for automated remediation
- Create an end-to-end auto-remediation pipeline using CloudWatch Alarm → EventBridge → Lambda
- Understand Lambda concurrency, timeout, and error handling patterns

_Reference: [Lambda Docs](https://docs.aws.amazon.com/lambda/latest/dg/) | [S3 Event Notifications](https://docs.aws.amazon.com/AmazonS3/latest/userguide/NotificationHowTo.html)_

## Event-Driven Architecture on AWS

```
Event Sources           Routing Layer          Consumers
────────────           ─────────────          ─────────
S3 (object created) ─→                    ─→ Lambda
RDS (event)         ─→  EventBridge        ─→ SQS queue
EC2 (state change)  ─→  (default bus)      ─→ SNS
CloudWatch Alarm    ─→                    ─→ Step Functions
Custom apps         ─→  Custom bus         ─→ Kinesis
```

## Lambda Key Concepts for CloudOps

| Setting | Default | Max | Notes |
|---------|---------|-----|-------|
| Timeout | 3s | 15 min | Set based on expected execution time + buffer |
| Memory | 128 MB | 10,240 MB | CPU scales linearly with memory |
| Concurrency | 1,000/region | Configurable | Reserved + provisioned concurrency |
| Payload (sync) | 6 MB | 6 MB | Async: 256 KB |

## S3 Event Notification Targets
- **Lambda**: Direct invocation (synchronous from S3 perspective)
- **SQS**: Buffering for high-volume events
- **SNS**: Fan-out to multiple subscribers
- **EventBridge**: Rich routing + filtering (recommended for new architectures)

## Auto-Remediation Pattern
```
EC2 CPU > 90% for 5min
  → CloudWatch Alarm (ALARM state)
  → EventBridge rule matches AlarmState=ALARM
  → Triggers Lambda function
  → Lambda checks instance metrics
  → If truly stuck: Lambda calls EC2 reboot API
  → Lambda creates OpsCenter OpsItem for tracking
  → SNS notification to on-call team
```

In [ ]:
import boto3, json, zipfile, io

lambda_client = boto3.client('lambda', region_name='us-east-1')
events = boto3.client('events', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
REGION = 'us-east-1'
print(f'Account: {account_id}')

In [ ]:
# Define a Lambda function for auto-remediation
# This function checks S3 bucket public access settings on creation

lambda_code = '''
import json
import boto3
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

def handler(event, context):
    """Auto-remediate S3 buckets that become public."""
    s3 = boto3.client('s3')
    
    # Extract bucket name from EventBridge event (from CloudTrail)
    detail = event.get('detail', {})
    bucket_name = detail.get('requestParameters', {}).get('bucketName', '')
    
    if not bucket_name:
        logger.warning('No bucket name found in event')
        return {'statusCode': 400, 'body': 'No bucket name in event'}
    
    logger.info(f'Checking public access block for bucket: {bucket_name}')
    
    try:
        # Check current public access block settings
        response = s3.get_public_access_block(Bucket=bucket_name)
        config = response['PublicAccessBlockConfiguration']
        
        all_blocked = all([
            config.get('BlockPublicAcls', False),
            config.get('IgnorePublicAcls', False),
            config.get('BlockPublicPolicy', False),
            config.get('RestrictPublicBuckets', False)
        ])
        
        if not all_blocked:
            # REMEDIATE: Re-enable public access block
            logger.warning(f'Bucket {bucket_name} has public access! Remediating...')
            s3.put_public_access_block(
                Bucket=bucket_name,
                PublicAccessBlockConfiguration={
                    'BlockPublicAcls': True,
                    'IgnorePublicAcls': True,
                    'BlockPublicPolicy': True,
                    'RestrictPublicBuckets': True
                }
            )
            logger.info(f'Remediated: Public access blocked on {bucket_name}')
            return {'statusCode': 200, 'body': f'Remediated {bucket_name}'}
        else:
            logger.info(f'Bucket {bucket_name} already has public access blocked')
            return {'statusCode': 200, 'body': 'Already compliant'}
            
    except Exception as e:
        logger.error(f'Error processing {bucket_name}: {str(e)}')
        raise
'''

# Create a ZIP package for Lambda deployment
buffer = io.BytesIO()
with zipfile.ZipFile(buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('lambda_function.py', lambda_code)
buffer.seek(0)
zip_bytes = buffer.read()

print(f'Lambda package size: {len(zip_bytes)} bytes')
print('Function: Auto-remediate S3 buckets with public access enabled')

In [ ]:
# Create the Lambda function
# Requires IAM role with S3 and Lambda permissions

# First, check if we have a suitable role
iam = boto3.client('iam')
ROLE_NAME = 'cloudops-lambda-remediation-role'

try:
    role = iam.get_role(RoleName=ROLE_NAME)
    ROLE_ARN = role['Role']['Arn']
    print(f'Using existing role: {ROLE_ARN}')
except iam.exceptions.NoSuchEntityException:
    # Create role
    trust = {'Version': '2012-10-17', 'Statement': [{
        'Effect': 'Allow', 'Principal': {'Service': 'lambda.amazonaws.com'},
        'Action': 'sts:AssumeRole'
    }]}
    role = iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust))
    ROLE_ARN = role['Role']['Arn']
    iam.attach_role_policy(RoleName=ROLE_NAME, PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole')
    iam.attach_role_policy(RoleName=ROLE_NAME, PolicyArn='arn:aws:iam::aws:policy/AmazonS3FullAccess')
    import time; time.sleep(10)  # Wait for role propagation
    print(f'Created role: {ROLE_ARN}')

FUNCTION_NAME = 'cloudops-s3-remediation'
try:
    response = lambda_client.create_function(
        FunctionName=FUNCTION_NAME,
        Runtime='python3.12',
        Role=ROLE_ARN,
        Handler='lambda_function.handler',
        Code={'ZipFile': zip_bytes},
        Description='Auto-remediate S3 buckets with public access',
        Timeout=30,
        MemorySize=128,
        Environment={'Variables': {'LOG_LEVEL': 'INFO'}}
    )
    print(f'Created Lambda: {response["FunctionArn"]}')
except lambda_client.exceptions.ResourceConflictException:
    print(f'Lambda {FUNCTION_NAME} already exists.')

## Exercise

In [ ]:
# TODO: Create an EventBridge rule that triggers the Lambda above when:
# - Source: aws.s3
# - Event: CreateBucket (via CloudTrail)
# Hint: This uses the same pattern as Lesson 2.3 (CloudTrail → EventBridge)
# The rule should filter for: source=[aws.s3], detail.eventName=[CreateBucket]

import boto3, json
events = boto3.client('events', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']

# YOUR CODE HERE
# 1. Create the EventBridge rule
# 2. Add Lambda as target
# 3. Grant EventBridge permission to invoke Lambda (lambda.add_permission)

In [ ]:
# Cleanup
import boto3
lambda_client = boto3.client('lambda', region_name='us-east-1')
events = boto3.client('events', region_name='us-east-1')
iam = boto3.client('iam')

for rule in ['S3BucketCreated']:
    try:
        targets = events.list_targets_by_rule(Rule=rule)['Targets']
        if targets:
            events.remove_targets(Rule=rule, Ids=[t['Id'] for t in targets])
        events.delete_rule(Name=rule)
        print(f'Deleted rule: {rule}')
    except Exception as e:
        print(f'{rule}: {e}')

try:
    lambda_client.delete_function(FunctionName='cloudops-s3-remediation')
    print('Deleted Lambda function')
except Exception as e:
    print(f'Lambda: {e}')

try:
    for policy in ['arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole',
                   'arn:aws:iam::aws:policy/AmazonS3FullAccess']:
        iam.detach_role_policy(RoleName='cloudops-lambda-remediation-role', PolicyArn=policy)
    iam.delete_role(RoleName='cloudops-lambda-remediation-role')
    print('Deleted IAM role')
except Exception as e:
    print(f'IAM: {e}')

## Summary
- **Event-driven automation** reacts to changes without polling — more efficient and faster than scheduled checks
- **S3 Event Notifications** → Lambda (direct), SQS (buffered), SNS (fan-out), EventBridge (rich routing)
- **Auto-remediation pipeline**: CloudWatch Alarm → EventBridge → Lambda → API action
- Lambda needs the right IAM role (principle of least privilege — only grant what's needed)
- Always set **timeout** based on expected max execution time; set **DLQ** for failures

---
**Next Phase:** [Phase 5 — Security & Compliance](../phase-5-security/lesson-5.1-iam-advanced.ipynb)